In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [2]:
spark = SparkSession.builder.appName("ReadJSON").getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/11 11:29:05 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/03/11 11:29:07 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [3]:
from pathlib import Path

# Resolve absolute path to data/raw from the notebook location
notebook_dir = Path().resolve()  # notebooks/
data_path = notebook_dir.parent / "data" / "raw"

In [4]:
print(f"Current dir: {notebook_dir}")
print(f"Data path: {data_path}")
print(f"Path exists: {data_path.exists()}")
print(f"JSON files found: {list(data_path.glob('*.json'))}")

Current dir: /Users/snehamungre/projects/crypto_market_analysis/notebooks
Data path: /Users/snehamungre/projects/crypto_market_analysis/data/raw
Path exists: True
JSON files found: [PosixPath('/Users/snehamungre/projects/crypto_market_analysis/data/raw/crypto_market_data_raw_2026-03-03_09-48-58.json'), PosixPath('/Users/snehamungre/projects/crypto_market_analysis/data/raw/crypto_market_data_raw_2026-03-10_10-07-44.json'), PosixPath('/Users/snehamungre/projects/crypto_market_analysis/data/raw/crypto_market_data_raw_2026-02-27_10-24-30.json'), PosixPath('/Users/snehamungre/projects/crypto_market_analysis/data/raw/crypto_market_data_raw_2026-03-09_12-53-48.json'), PosixPath('/Users/snehamungre/projects/crypto_market_analysis/data/raw/crypto_market_data_raw_2026-02-26_08-43-05.json'), PosixPath('/Users/snehamungre/projects/crypto_market_analysis/data/raw/crypto_market_data_raw_2026-03-05_14-02-47.json'), PosixPath('/Users/snehamungre/projects/crypto_market_analysis/data/raw/crypto_market_d

In [5]:
from pyspark.sql.types import *

schema = StructType(
    [
        StructField("ath", DoubleType(), True),
        StructField("ath_change_percentage", DoubleType(), True),
        StructField("ath_date", StringType(), True),
        StructField("atl", DoubleType(), True),
        StructField("atl_change_percentage", DoubleType(), True),
        StructField("atl_date", StringType(), True),
        StructField("circulating_supply", DoubleType(), True),
        StructField("current_price", DoubleType(), True),
        StructField("fully_diluted_valuation", LongType(), True),
        StructField("high_24h", DoubleType(), True),
        StructField("id", StringType(), True),
        StructField("image", StringType(), True),
        StructField("last_updated", StringType(), True),
        StructField("low_24h", DoubleType(), True),
        StructField("market_cap", LongType(), True),
        StructField("market_cap_change_24h", DoubleType(), True),
        StructField("market_cap_change_percentage_24h", DoubleType(), True),
        StructField("market_cap_rank", LongType(), True),
        StructField("max_supply", DoubleType(), True),
        StructField("name", StringType(), True),
        StructField("price_change_24h", DoubleType(), True),
        StructField("price_change_percentage_24h", DoubleType(), True),
        StructField(
            "roi",
            StructType(
                [
                    StructField("currency", StringType(), True),
                    StructField("percentage", DoubleType(), True),
                    StructField("times", DoubleType(), True),
                ]
            ),
            True,
        ),
        StructField(
            "sparkline_in_7d",
            StructType(
                [
                    StructField("price", ArrayType(DoubleType()), True),
                ]
            ),
            True,
        ),
        StructField("symbol", StringType(), True),
        StructField("total_supply", DoubleType(), True),
        StructField("total_volume", DoubleType(), True),
    ]
)

In [6]:
from pyspark.sql.functions import to_timestamp

df = (
    spark.read.option("multiLine", True)
    .option("header", True)
    .schema(schema)
    .json(str(data_path))
)

df = (
    df.withColumn("last_updated", to_timestamp("last_updated"))
    .withColumn("ath_date", to_timestamp("ath_date"))
    .withColumn("atl_date", to_timestamp("atl_date"))
    .drop("image", "symbol", "roi")
)

df.show(7)
df.count()

+--------+---------------------+--------------------+----------+---------------------+--------------------+--------------------+-------------+-----------------------+--------+-----------+--------------------+--------+-------------+---------------------+--------------------------------+---------------+----------+--------+--------------------+---------------------------+--------------------+--------------------+---------------+
|     ath|ath_change_percentage|            ath_date|       atl|atl_change_percentage|            atl_date|  circulating_supply|current_price|fully_diluted_valuation|high_24h|         id|        last_updated| low_24h|   market_cap|market_cap_change_24h|market_cap_change_percentage_24h|market_cap_rank|max_supply|    name|    price_change_24h|price_change_percentage_24h|     sparkline_in_7d|        total_supply|   total_volume|
+--------+---------------------+--------------------+----------+---------------------+--------------------+--------------------+------------

1000

In [7]:
df.printSchema()

root
 |-- ath: double (nullable = true)
 |-- ath_change_percentage: double (nullable = true)
 |-- ath_date: timestamp (nullable = true)
 |-- atl: double (nullable = true)
 |-- atl_change_percentage: double (nullable = true)
 |-- atl_date: timestamp (nullable = true)
 |-- circulating_supply: double (nullable = true)
 |-- current_price: double (nullable = true)
 |-- fully_diluted_valuation: long (nullable = true)
 |-- high_24h: double (nullable = true)
 |-- id: string (nullable = true)
 |-- last_updated: timestamp (nullable = true)
 |-- low_24h: double (nullable = true)
 |-- market_cap: long (nullable = true)
 |-- market_cap_change_24h: double (nullable = true)
 |-- market_cap_change_percentage_24h: double (nullable = true)
 |-- market_cap_rank: long (nullable = true)
 |-- max_supply: double (nullable = true)
 |-- name: string (nullable = true)
 |-- price_change_24h: double (nullable = true)
 |-- price_change_percentage_24h: double (nullable = true)
 |-- sparkline_in_7d: struct (nullable

In [8]:
filter_cols = [
    f.name
    for f in df.schema.fields
    if ("Double" in str(f.dataType) or "Long" in str(f.dataType))
    and ("change" not in f.name and "percentage" not in f.name)
]

print(filter_cols)

['ath', 'atl', 'circulating_supply', 'current_price', 'fully_diluted_valuation', 'high_24h', 'low_24h', 'market_cap', 'market_cap_rank', 'max_supply', 'sparkline_in_7d', 'total_supply', 'total_volume']


In [9]:
from pyspark.sql import functions as F

# Access the column and field using F.col
price_array = F.col("sparkline_in_7d.price")

# finds mean of the 7 day prices obtained
df = df.withColumn(
    "7d_avg",
    F.aggregate(price_array, F.lit(0.0), lambda acc, x: acc + x) / F.size(price_array),
)

# finds high of the 7 day prices obtained
df = df.withColumn("7d_max", F.array_max(price_array))

# finds low of the 7 day prices obtained
df = df.withColumn("7d_low", F.array_min(price_array))


df.select("7d_low", "7d_avg", "7d_max").show(5, truncate=False)

df = df.drop("sparkline_in_7d")

+------------------+------------------+------------------+
|7d_low            |7d_avg            |7d_max            |
+------------------+------------------+------------------+
|63176.9334493685  |68116.16727327193 |73669.77886917477 |
|1841.2881843632538|2002.0260952556205|2179.3034097539403|
|0.9996604391196411|1.0000431409656692|1.0004365229099252|
|591.2802216239186 |630.0843376260655 |664.0544992300274 |
|1.2779557957398355|1.3797567174909626|1.4591620419556068|
+------------------+------------------+------------------+
only showing top 5 rows
